# Text Representation

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [3]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [4]:
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [7]:
tfidf_vectorizer = TfidfVectorizer() # Create a CountVectorizer instance
vector_matrix = tfidf_vectorizer.fit_transform(data) # Fit and transform the documents into numerical vectors

In [8]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


In [9]:
import pandas as pd
import numpy as np
import gensim


In [10]:
from pathlib import Path
import pandas as pd
import re
from gensim.models import Word2Vec

simpsons_path = Path("/home/ubuntu/upload/simpsons_script_lines.csv")
if not simpsons_path.exists():
    simpsons_path = Path("simpsons_script_lines.csv")
df = pd.read_csv(simpsons_path, usecols=["spoken_words"]).dropna()
print("Number of Simpsons lines:", len(df))
df.head()


Number of Simpsons lines: 131855


,spoken_words
0,"No, actually, it was a little of both. Sometim..."
1,Where's Mr. Bergstrom?
2,I don't know. Although I'd sure like to talk t...
3,That life is worth living.
4,The polls will be open from now until the end ...


In [11]:
def tokenize_text(text):
    return re.findall(r"[a-z]+", str(text).lower())

tokens = [tokenize_text(text) for text in df["spoken_words"]]
tokens = [sentence for sentence in tokens if sentence]
print("Tokenized lines:", len(tokens))


Tokenized lines: 131817


In [12]:
w2v = Word2Vec(
    sentences=tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1,
    workers=1,
    seed=42,
    epochs=5
)
print("Word2Vec vocabulary size:", len(w2v.wv))


Word2Vec vocabulary size: 38591


In [13]:
print("Cosine similarity between 'homer' and 'marge':", w2v.wv.similarity("homer", "marge"))


Cosine similarity between 'homer' and 'marge': 0.8582513


In [14]:
print("Words similar to 'homer':", w2v.wv.most_similar("homer", topn=5))


Words similar to 'homer': [('abe', 0.8620306849479675), ('marge', 0.8582512736320496), ('eliza', 0.8216102123260498), ('grampa', 0.8203791379928589), ('barney', 0.8196417093276978)]


# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


In [15]:
# Task 1: Cosine Similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sentences = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?"
]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(sentences)
similarity_matrix = cosine_similarity(X)
print(pd.DataFrame(similarity_matrix, index=["S1", "S2", "S3", "S4"], columns=["S1", "S2", "S3", "S4"]).round(3))


       S1     S2     S3     S4
S1  1.000  0.647  0.308  1.000
S2  0.647  1.000  0.225  0.647
S3  0.308  0.225  1.000  0.308
S4  1.000  0.647  0.308  1.000


### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


In [16]:
# Task 2: TF-IDF and important words
sentences = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data"
]
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(sentences)
terms = tfidf.get_feature_names_out()
for row_number, row in enumerate(X.toarray(), start=1):
    ranked = sorted(zip(terms, row), key=lambda item: item[1], reverse=True)
    print(f"Sentence {row_number} important words:")
    print([(word, round(score, 3)) for word, score in ranked if score > 0][:5])


Sentence 1 important words:
[('of', np.float64(0.488)), ('science', np.float64(0.488)), ('fields', np.float64(0.321)), ('important', np.float64(0.321)), ('most', np.float64(0.321))]
Sentence 2 important words:
[('best', np.float64(0.4)), ('courses', np.float64(0.4)), ('this', np.float64(0.4)), ('is', np.float64(0.304)), ('of', np.float64(0.304))]
Sentence 3 important words:
[('data', np.float64(0.641)), ('analyze', np.float64(0.543)), ('scientists', np.float64(0.543))]


## Word2vec

### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)

In [17]:
# Task 3: Preprocess Simpsons spoken_words and train Skip-gram Word2Vec
import re
from gensim.models import Word2Vec

simpsons = pd.read_csv(simpsons_path, usecols=["spoken_words"]).dropna()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,”“.’$-]", "", text)
    return text

cleaned_lines = simpsons["spoken_words"].apply(clean_text)
simpsons_tokens = [re.findall(r"[a-z]+", line) for line in cleaned_lines]
simpsons_tokens = [line for line in simpsons_tokens if line]
Skip_gram_model = Word2Vec(
    simpsons_tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1,
    workers=1,
    seed=42,
    epochs=5
)
print("Cleaned lines:", len(simpsons_tokens))
print("Skip-gram vocabulary size:", len(Skip_gram_model.wv))


Cleaned lines: 131817
Skip-gram vocabulary size: 43784


### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



In [18]:
# Task 4: Words similar to Homer, Marge, and Bart
for name in ["homer", "marge", "bart"]:
    print(f"Words similar to {name}:")
    print(Skip_gram_model.wv.most_similar(name, topn=5))


Words similar to homer:
[('marge', 0.8653039336204529), ('abe', 0.8517967462539673), ('bartholomew', 0.8321748971939087), ('grampa', 0.8267911076545715), ('eliza', 0.8233173489570618)]
Words similar to marge:
[('homer', 0.8653038740158081), ('abe', 0.8520047664642334), ('sweetie', 0.8237380981445312), ('becky', 0.8220895528793335), ('eliza', 0.8120144009590149)]
Words similar to bart:
[('milhouse', 0.8608148097991943), ('lisa', 0.853973388671875), ('grampa', 0.8382692337036133), ('abe', 0.8268643021583557), ('eliza', 0.8228848576545715)]


### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

In [19]:
# Task 5: Words that do not belong to each list
word_lists = [
    ["jimbo", "milhouse", "kearney"],
    ["nelson", "bart", "milhouse"],
    ["homer", "patty", "selma"]
]
for words in word_lists:
    print(f"{words} -> odd word: {Skip_gram_model.wv.doesnt_match(words)}")


['jimbo', 'milhouse', 'kearney'] -> odd word: milhouse
['nelson', 'bart', 'milhouse'] -> odd word: nelson
['homer', 'patty', 'selma'] -> odd word: homer
